# Train Models with Levanter (from README)

This notebook follows the steps from the README to install Levanter and run training commands.

It demonstrates:
- Installing Levanter in editable mode
- Training a GPT-2 "nano" model
- Training a Llama-small model on OpenWebText with configuration overrides

Notes:
- If you use Weights & Biases (W&B), run `wandb login` before training.
- GPU/TPU setup is platform-specific; see docs for GPU/TPU guides.
- This notebook assumes it is located in the `notebooks/` folder of the repo.

## 0) Setup: clone repo to /content/levanter
Mirrors style_prefix_demo: if present, fetch branch; else clone.

In [ ]:
%%bash
set -euo pipefail
REPO_DIR=/content/levanter
REPO_URL=https://github.com/chris544460/levanter.git
BRANCH=feat/style-prefix-token
if [ -d "$REPO_DIR/.git" ]; then
  echo "Found repo at $REPO_DIR; syncing $BRANCH"
  git -C "$REPO_DIR" fetch origin "$BRANCH"
  git -C "$REPO_DIR" checkout "$BRANCH" || true
  git -C "$REPO_DIR" reset --hard "origin/$BRANCH"
  git -C "$REPO_DIR" clean -fd
else
  echo "Cloning $REPO_URL to $REPO_DIR"
  rm -rf "$REPO_DIR"
  git clone "$REPO_URL" "$REPO_DIR"
  git -C "$REPO_DIR" checkout "$BRANCH" || true
  git -C "$REPO_DIR" reset --hard "origin/$BRANCH" || true
fi


## 1) Install Levanter (editable)
Per the README, after installing JAX for your platform, install Levanter.
If running from this repo, editable install keeps changes live.

In [ ]:
%%bash
set -euo pipefail
cd /content/levanter

# Ensure compatible/proper deps like draccus and protobuf
python -m pip uninstall -y protobuf || true
python -m pip install --upgrade "protobuf>=5.26.1,<6"
python -m pip install --upgrade "draccus>=0.11.5"
python -m pip show draccus | sed -n '1,5p'
python -m pip install --upgrade "equinox<0.13,!=0.12.0"
python -m pip install --upgrade "haliax>=1.4.dev404"
python -m pip install --upgrade "equinox<0.13,!=0.12.0"

# Editable install from the current repo (no deps to preserve pinned versions)
pip uninstall -y levanter || true
python -m pip install -e . --no-deps


Optional: Enable JAX GPU (CUDA 12)

If running with a CUDA 12 GPU runtime (e.g., Colab), install the matching JAX CUDA plugin to avoid PJRT/plugin mismatches.

In [ ]:
# Uncomment to enable GPU JAX on CUDA12 runtimes
# !pip install --upgrade "jax[cuda12]==0.7.2"


## 2) Verify imports and devices

In [ ]:
import os
# Force CPU to avoid Colab GPU plugin/version mismatches.
os.environ.setdefault('JAX_PLATFORMS', 'cpu')
import jax
import levanter
print('JAX devices:', jax.devices())
print('Levanter version:', getattr(levanter, '__version__', 'dev'))
print('JAX_PLATFORMS=', os.environ.get('JAX_PLATFORMS'))

## 3) Train a GPT-2 "nano" (from README)
This mirrors the README's quickstart. It trains on WikiText-103 by default.

In [ ]:
cmd = 'cd /content/levanter && python -m levanter.main.train_lm --config_path /content/levanter/config/gpt2_nano.yaml --trainer.tracker "{type: noop}" --trainer.log_jaxprs false --trainer.log_xla_hlo false'
print(cmd)
!{cmd}


## 3b) Inference from latest checkpoint (GPT-2 nano)

Find the most recent checkpoint under `checkpoints/` and sample a completion using the GPT-2 nano config.
If you just ran section 3), this should pick up that run.

In [ ]:
%%bash
set -euo pipefail
REPO_DIR=/content/levanter
cd "$REPO_DIR"

LATEST=$(python - <<'PY'
import os, json, glob
cands = glob.glob(os.path.join('checkpoints', '*', 'step-*'))
cands = [p for p in cands if os.path.exists(os.path.join(p, 'metadata.json'))]
def score(p):
    try:
        with open(os.path.join(p, 'metadata.json')) as f:
            m = json.load(f)
        return (m.get('timestamp',''), int(m.get('step', -1)))
    except Exception:
        return ('', -1)
if not cands:
    raise SystemExit('No checkpoints found under checkpoints/')
print(sorted(cands, key=score)[-1])
PY
)
echo "Latest checkpoint: $LATEST"

# GPT-2 nano model dims must match training config
python -m levanter.main.sample_lm \
  --checkpoint_path "$LATEST" \
  --tokenizer gpt2 \
  --model.type gpt2 \
  --model.hidden_dim 32 \
  --model.num_heads 4 \
  --model.num_layers 2 \
  --temperature 0.0 \
  --max_new_tokens 16 \
  --prompts "What is the capital of France?"


## 4) Train a Llama-small on your own data (from README)
You can override dataset fields via CLI flags. Here we set `--data.id openwebtext`.
Optionally, you may set a tokenizer and/or a cache directory.

In [ ]:
cmd = 'cd /content/levanter && python -m levanter.main.train_lm --config_path /content/levanter/config/llama_small_fast.yaml --data.id openwebtext'
print(cmd)
!{cmd}


## 4b) Inference from latest checkpoint (Llama-small)

If you ran section 4), this will sample from the most recent Llama-small checkpoint.
Tokenizer and model dims mirror `config/llama_small_fast.yaml`.

In [ ]:
%%bash
set -euo pipefail
REPO_DIR=/content/levanter
cd "$REPO_DIR"

LATEST=$(python - <<'PY'
import os, json, glob
cands = glob.glob(os.path.join('checkpoints', '*', 'step-*'))
cands = [p for p in cands if os.path.exists(os.path.join(p, 'metadata.json'))]
def score(p):
    try:
        with open(os.path.join(p, 'metadata.json')) as f:
            m = json.load(f)
        return (m.get('timestamp',''), int(m.get('step', -1)))
    except Exception:
        return ('', -1)
if not cands:
    raise SystemExit('No checkpoints found under checkpoints/')
print(sorted(cands, key=score)[-1])
PY
)
echo "Latest checkpoint: $LATEST"

python -m levanter.main.sample_lm \
  --checkpoint_path "$LATEST" \
  --tokenizer NousResearch/Llama-2-7b-hf \
  --model.type llama \
  --model.hidden_dim 768 \
  --model.intermediate_dim 2048 \
  --model.num_heads 12 \
  --model.num_kv_heads 12 \
  --model.num_layers 12 \
  --model.seq_len 1024 \
  --temperature 0.0 \
  --max_new_tokens 16 \
  --prompts "What is the capital of France?"


In [ ]:
# Example with tokenizer and cache dir (edit and uncomment to use):
# !python -m levanter.main.train_lm \
#   --config_path /content/levanter/config/llama_small_fast.yaml \
#   --data.id openwebtext \
#   --data.tokenizer "NousResearch/Llama-2-7b-hf" \
#   --data.cache_dir "gs://path/to/cache/dir"

## 5) Customize a config (reference)
Edit YAML under `config/` to change model and training params. For reference, here's the `llama_small_fast.yaml` mentioned in the README.

In [ ]:
from pathlib import Path
print(Path('/content/levanter/config/llama_small_fast.yaml').read_text())